In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
)

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

from rag_helper import RAGBase

assistant = RAGBase(index=index, llm_client=openai_client)
query = "I just discovered the course. Can I still enroll?"

assistant.rag(query)


'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [2]:

class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict,
        )


In [3]:

texts = []
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)


from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

import numpy as np

x = np.array(vectors)

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(x, documents)

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

In [4]:

answer = vector_assistant.rag("the program has already begun, can I still sign up?")
print(answer)


Yes, you can still join. If you want a certificate, make sure you submit your project while submissions are still open.


In [5]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)


In [7]:
vs_index.clear()
vs_index.fit(vectors, documents)

In [8]:
query = "the program has already begun, can I still sign up?"
query_vector = model.encode(query)

In [10]:
results = vs_index.search(query_vector, num_results=5, filter_dict={'course': 'llm-zoomcamp'})
print(results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': 'bd31146b0e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'When will the course be offered next?', 'answer': 'Summer 2027.'}, {'id': '897a350476', 'course': 'llm-zoomcamp', 'section': 'Cap

In [11]:
vs_index.close()